# SadTalker API pour EduAvatar (avec ngrok)

**Repos officiels :**
- SadTalker : https://github.com/OpenTalker/SadTalker
- Notebook officiel : https://colab.research.google.com/github/Winfredy/SadTalker/blob/main/quick_demo.ipynb
- Démo en ligne (HuggingFace) : https://huggingface.co/spaces/vinthony/SadTalker

> **IMPORTANT :** Activez le GPU dans **Runtime → Change runtime type → GPU** AVANT de lancer.
> Ce notebook clone le code OFFICIEL et expose une API via ngrok qui sera appelée par votre backend EduAvatar (`sadtalker.service.js`).

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 2. Installation (environ 5 min)

Cellule OFFICIELLE du repo SadTalker (merci aux auteurs).

In [ ]:
# ===== Installation officielle SadTalker =====

# Force python 3.8 (requis par les dépendances du repo)
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.8 2
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.9 1
!sudo apt install python3.8
!sudo apt-get install python3.8-distutils
!python --version

!apt-get update
!apt install software-properties-common
!sudo dpkg --remove --force-remove-reinstreq python3-pip python3-setuptools python3-wheel
!apt-get install python3-pip

# Clone le repo officiel
print('Clone du projet SadTalker officiel...')
!git clone https://github.com/OpenTalker/SadTalker.git
%cd /content/SadTalker
!export PYTHONPATH=/content/SadTalker:$PYTHONPATH

# PyTorch compatible CUDA (pinned par les auteurs)
!python3.8 -m pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113

# FFmpeg pour l'encodage vidéo
!apt update
!apt install ffmpeg

# Dépendances du projet
!python3.8 -m pip install -r requirements.txt

## 3. Télécharger les modèles pré-entraînés (environ 1 min)

In [ ]:
# Script OFFICIEL de téléchargement des checkpoints
print('Téléchargement des modèles pré-entraînés...')
!rm -rf checkpoints
!bash scripts/download_models.sh

## 4. Installer Flask + ngrok (pour exposer l'API)

In [ ]:
!python3.8 -m pip install flask flask-cors pyngrok

## 5. Configurer votre token ngrok

Créez un compte gratuit : https://dashboard.ngrok.com → **Your Authtoken**

Collez votre token dans la variable ci-dessous.

In [ ]:
NGROK_TOKEN = "VOTRE_TOKEN_NGROK"  # <-- COLLEZ VOTRE TOKEN ICI

from pyngrok import ngrok
if NGROK_TOKEN != "VOTRE_TOKEN_NGROK":
    ngrok.set_auth_token(NGROK_TOKEN)
print("Token ngrok configuré" if NGROK_TOKEN != "VOTRE_TOKEN_NGROK" else "⚠️  Collez votre token puis relancez cette cellule")

## 6. Serveur Flask : API de génération

Cette API utilise la **CLI officielle `inference.py`** du repo (pas une API inventée) :

`POST /generate` (multipart) → fichiers `image` + `audio` → renvoie la **vidéo MP4** de l'avatar parlant.

In [ ]:
import os, glob, subprocess, tempfile
from flask import Flask, request, send_file, jsonify
from flask_cors import CORS

PYTHON_EXEC = 'python3.8'
BASE_DIR    = '/content/SadTalker'
RESULT_DIR  = os.path.join(BASE_DIR, 'results_api')
os.makedirs(RESULT_DIR, exist_ok=True)

app = Flask(__name__)
CORS(app)

@app.route('/health', methods=['GET'])
def health():
    """Vérifie que l'API fonctionne : GET https://<ngrok-url>/health"""
    return jsonify({"status": "ok", "message": "SadTalker API opérationnelle"})

@app.route('/generate', methods=['POST'])
def generate():
    """Reçoit une image + un audio, renvoie la vidéo de l'avatar parlant."""
    tmp = tempfile.mkdtemp()
    try:
        image = request.files.get('image')
        audio = request.files.get('audio')
        if image is None or audio is None:
            return jsonify({"error": "Champs 'image' et 'audio' requis (multipart/form-data)"}), 400

        # Sauvegarde des fichiers temporaires
        img_ext = os.path.splitext(image.filename)[1].lower()
        if img_ext not in ('.jpg', '.jpeg', '.png'):
            img_ext = '.png'
        img_path = os.path.join(tmp, 'source' + img_ext)
        aud_path = os.path.join(tmp, 'driven.wav')
        image.save(img_path)
        audio.save(aud_path)

        print(f"Image : {img_path}")
        print(f"Audio : {aud_path}")

        # ==== APPEL CLI OFFICIELLE SadTalker ====
        cmd = [
            PYTHON_EXEC, 'inference.py',
            '--source_image', img_path,
            '--driven_audio', aud_path,
            '--result_dir', RESULT_DIR,
            '--still',
            '--preprocess', 'crop',
            '--enhancer', 'gfpgan',
            '--size', '256'
        ]
        print('Commande :', ' '.join(cmd))

        proc = subprocess.run(
            cmd,
            cwd=BASE_DIR,
            capture_output=True,
            text=True,
            timeout=600  # la génération GPU peut prendre plusieurs minutes
        )
        print(proc.stdout[-3000:])
        if proc.returncode != 0:
            return jsonify({"error": proc.stderr[-3000:]}), 500

        # Récupère la vidéo générée la plus récente
        videos = glob.glob(os.path.join(RESULT_DIR, '**', '*.mp4'), recursive=True)
        if not videos:
            return jsonify({"error": "Aucune vidéo générée"}), 500

        video_path = max(videos, key=os.path.getmtime)
        print('Vidéo :', video_path)
        return send_file(video_path, mimetype='video/mp4')
    except subprocess.TimeoutExpired:
        return jsonify({"error": "Délai dépassé (600s)"}), 504
    except Exception as e:
        return jsonify({"error": str(e)}), 500


## 7. Lancer l'API + tunnel ngrok

Copiez l'URL affichée dans votre fichier `.env` : `SADTALKER_API_URL=<votre-url>`

In [ ]:
# Ouvre le tunnel ngrok puis démarre Flask dans un thread
import threading, time

public_url = ngrok.connect(5000)
print('==============================================================')
print('  VOTRE URL NGROK (à copier dans le .env du projet) :')
print(f'  SADTALKER_API_URL={public_url}')
print('  Test rapide :', str(public_url) + '/health')
print('==============================================================')

# Teste l'endpoint /health
import urllib.request
try:
    with urllib.request.urlopen(str(public_url) + '/health', timeout=15) as resp:
        print('Réponse /health :', resp.read().decode())
except Exception as e:
    print('Attente du serveur Flask...', e)

# Démarre Flask en arrière-plan
threading.Thread(
    target=app.run,
    kwargs={'host': '0.0.0.0', 'port': 5000, 'debug': False, 'use_reloader': False},
    daemon=True
).start()

print('\nLe serveur Flask tourne sur le port 5000 (tunnel ngrok actif).')


## 8. Test manuel depuis votre ordinateur

Une fois l'URL ngrok affichée (ex: `https://abc-123.ngrok-free.app`), ouvrez un terminal sur VOTRE MACHINE :

```bash
# Vérifier que l'API est joignable
curl https://abc-123.ngrok-free.app/health

# Générer une vidéo de test
curl -X POST https://abc-123.ngrok-free.app/generate \
  -F "image=@mon_avatar.jpg" \
  -F "audio=@voix.wav" \
  -o avatar_video.mp4
```

Puis redémarrez le backend EduAvatar pour qu'il utilise la nouvelle URL :
```bash
cd eduvatar-pfa
# mettre SADTALKER_API_URL dans .env puis :
docker-compose restart backend
```